In [ ]:
import os
os.environ["GROQ_API_KEY"] = ""

## Install libraries

In [ ]:
!pip install -q youtube-transcript-api langchain-community langchain-groq \
               faiss-cpu tiktoken python-dotenv

In [ ]:
!pip install -U langchain-text-splitters

In [ ]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_groq import ChatGroq
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate


## Step 1a - Indexing (Document Ingestion)

In [ ]:
video_id = "OWW1R50YiPE"

try:
    transcript_list = YouTubeTranscriptApi().fetch(video_id)


    transcript = " ".join(chunk.text for chunk in transcript_list)

    import re
    transcript = re.sub(r">>", "", transcript)

    print(transcript)

except TranscriptsDisabled:
    print("No captions available for this video.")

## Step 1b - Indexing (Text Splitting)

In [ ]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000 , chunk_overlap = 200)
chunks = splitter.create_documents([transcript])

In [ ]:
len(chunks)

In [ ]:
chunks[32]

## Step 1c & 1d - Indexing (Embedding Generation and Storing in Vector Store)

In [ ]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vector_store = FAISS.from_documents(chunks, embeddings)

In [ ]:
vector_store.index_to_docstore_id

In [ ]:
vector_store.get_by_ids(['01c6ffb6-9bd3-4c29-8856-e96e45a05013'])

## Step 2 - Retrieval

In [ ]:
retriever = vector_store.as_retriever(search_type = "similarity" , search_kwargs={"k":4})

In [ ]:
retriever

## Step 3 - Augmentation

In [ ]:
llm = ChatGroq(
    model_name="llama-3.1-8b-instant",
    temperature=0.2
)

In [ ]:
prompt = PromptTemplate(
    template = """
     you are a helpful assistant.
     Answer ONLY from the provided transcript context.
     If the context is insufficient, just say you don't know.

     {context}
     Question: {question}

     """,
    input_variables = ['context','question']
)

In [ ]:
question = "Do missy stole makeup?"
retrived_docs = retriever.invoke(question)

In [ ]:
retrived_docs

In [ ]:
context_text = "\n\n".join(doc.page_content for doc in retrived_docs)

In [ ]:
context_text

In [ ]:
final_prompt = prompt.format(context = context_text , question = question)

In [ ]:
final_prompt

## Step 4 - Generation

In [ ]:
answer = llm.invoke(final_prompt)
print(answer)

## Building a Chain

In [ ]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [ ]:
def format_docs(retrieved_docs):
  context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
  return context_text

In [ ]:
parallel_chain = RunnableParallel({
    'context': retriever | RunnableLambda(format_docs),
    'question': RunnablePassthrough()
})

In [ ]:
parallel_chain.invoke('Do missy stole makeup?')

In [ ]:
parser = StrOutputParser()

In [ ]:
main_chain = parallel_chain | prompt | llm | parser

In [ ]:
main_chain.invoke('Do missy stole makeup?')